# 02 · Wikipedia — LLM-built knowledge graph + Text2Cypher

An LLM extracted a typed knowledge graph from Wikipedia into AgensGraph. This
notebook tours the graph and asks questions in natural language — the LLM writes
the Cypher.

> Run `build.py` first to build the `wikipedia_kg` graph.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))  # examples/demos
from _common import agens, config
from _common.models import EMBED_DIM, get_embed_model, get_llm, configure_settings
config.require_openai_key()  # reads OPENAI_API_KEY from examples/demos/.env

configure_settings()  # Settings.llm / Settings.embed_model
store = agens.make_pg_store("wikipedia_kg", vector_dimension=EMBED_DIM,
                            enhanced_schema=True, create=False)
llm = get_llm()

## The extracted graph

Entity counts by type, and a few example relationships.

In [2]:
import pandas as pd
comp = store.structured_query('''MATCH (n:"__Node__") UNWIND n.labels AS t
    WITH t WHERE t <> '__Entity__' AND t <> 'Chunk'
    RETURN t AS type, count(*) AS n ORDER BY n DESC''')
display(pd.DataFrame(comp))
rels = store.structured_query('''MATCH (a:"__Node__")-[r]->(b:"__Node__")
    WHERE '__Entity__' IN a.labels AND '__Entity__' IN b.labels
    RETURN a.name AS source, type(r) AS rel, b.name AS target LIMIT 8''')
pd.DataFrame(rels)

,type,n
0,WORK,1383
1,EVENT,1170
2,PLACE,975
3,PERSON,892
4,ORGANIZATION,635
5,ENTITY,161
6,DATE,112
7,ELEMENT,8
8,FIELD,5
9,CONCEPT,5


,source,rel,target
0,Anarchism,CREATED,Anarchist Movement
1,Albedo,CREATED,Surface albedo
2,A,CREATED,Latin alphabet
3,aleph,CREATED,Phoenician alphabet
4,Alabama,CREATED,Recognized as a state in December 1819
5,Achilles,CREATED,Achilleid
6,Abraham Lincoln,CREATED,U.S. federal government
7,Aristotle,CREATED,many of his hundreds of books


## The schema fed to Text2Cypher

`enhanced_schema=True` adds example values, helping the LLM write good Cypher.

In [3]:
print(store.get_schema_str()[:1200])

Node properties:
CONCEPT {embedding: LIST, id: STRING, name: STRING, title: STRING, triplet_source_id: STRING, url: STRING}
CONSTELLATION {embedding: LIST, id: STRING, name: STRING, title: STRING, triplet_source_id: STRING, url: STRING}
Chunk {_node_content: STRING, _node_type: STRING, doc_id: STRING, document_id: STRING, embedding: LIST, id: STRING, ref_doc_id: STRING, text: STRING, title: STRING, url: STRING}
DATE {embedding: LIST, id: STRING, name: STRING, title: STRING, triplet_source_id: STRING, url: STRING}
ELEMENT {embedding: LIST, id: STRING, name: STRING, title: STRING, triplet_source_id: STRING, url: STRING}
ENTITY {embedding: LIST, id: STRING, name: STRING, title: STRING, triplet_source_id: STRING, url: STRING}
EVENT {embedding: LIST, id: STRING, name: STRING, title: STRING, triplet_source_id: STRING, url: STRING}
FIELD {embedding: LIST, id: STRING, name: STRING, title: STRING, triplet_source_id: STRING, url: STRING}
MOVEMENT {embedding: LIST, id: STRING, name: STRING, title

## Text2Cypher — natural language → AgensGraph Cypher

The store ships an AgensGraph-dialect prompt, so a plain `TextToCypherRetriever`
generates runnable Cypher. `SafeTextToCypherRetriever` (in `_common/cypher.py`)
adds error-isolation so a bad generation can't crash the query.

In [4]:
from _common.cypher import SafeTextToCypherRetriever, read_only_validator
t2c = SafeTextToCypherRetriever(graph_store=store, llm=llm, cypher_validator=read_only_validator)
nodes = t2c.retrieve("How many entities of each type are there?")
print(nodes[0].node.text if nodes else "(no result)")

Generated Cypher query:
MATCH (n:"__Node__") UNWIND n.labels AS t WITH t WHERE t <> '__Entity__'
RETURN t AS type, count(*) AS n ORDER BY n DESC LIMIT 50

Cypher Response:
[{'type': 'WORK', 'n': 1383}, {'type': 'EVENT', 'n': 1170}, {'type': 'PLACE', 'n': 975}, {'type': 'PERSON', 'n': 892}, {'type': 'ORGANIZATION', 'n': 635}, {'type': 'Chunk', 'n': 500}, {'type': 'ENTITY', 'n': 161}, {'type': 'DATE', 'n': 112}, {'type': 'ELEMENT', 'n': 8}, {'type': 'FIELD', 'n': 5}, {'type': 'CONCEPT', 'n': 5}, {'type': 'CONSTELLATION', 'n': 4}, {'type': 'YEAR', 'n': 1}, {'type': 'MOVEMENT', 'n': 1}, {'type': 'PHILOSOPHY', 'n': 1}]


## Full retriever stack

Combine keyword (`LLMSynonymRetriever`), vector (`VectorContextRetriever`) and
Text2Cypher retrieval behind one query engine.

In [5]:
from llama_index.core import PropertyGraphIndex
from llama_index.core.indices.property_graph import LLMSynonymRetriever, VectorContextRetriever
index = PropertyGraphIndex.from_existing(store, embed_model=get_embed_model(), llm=llm,
                                         kg_extractors=[], use_async=False)
qe = index.as_query_engine(sub_retrievers=[
    LLMSynonymRetriever(graph_store=store, llm=llm, include_text=True),
    VectorContextRetriever(graph_store=store, embed_model=get_embed_model(),
                           similarity_top_k=5, path_depth=1, include_text=True),
    SafeTextToCypherRetriever(graph_store=store, llm=llm, cypher_validator=read_only_validator),
])
print(qe.query("Which 5 entities are connected to the most others?"))

The five entities connected to the most others are:

1. 13 Colonial States
2. 2-aminopropanoic acid
3. 37,500 main lines in use
4. A. kadabba
5. 110,200 mobile cellular


## How it was built

`build.py` extracts the graph with one call:

```python
extractor = SchemaLLMPathExtractor(llm=llm,
    possible_entities=Literal["Person","Organization","Place","Event","Work"],
    possible_relations=Literal["FOUNDED","LOCATED_IN","BORN_IN", ...], strict=False)
PropertyGraphIndex.from_documents(docs, property_graph_store=store,
    kg_extractors=[extractor], embed_model=embed, llm=llm)
```

In [6]:
agens.close()